# R08-H56 - Proposition coverage audit (deterministic half only)

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R08 conformist round, coverage audit <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only) <br>

Registered deterministic clause: fraction of evidence-bearing chunks (chunks whose text contains any of
the 33 gold strings) that carry ZERO propositions derived from them. Bar: >=20% gaps confirms the
coverage clause; <5% refutes (already saturated). The generation half is NOT run.

Provenance caveat: Proposition nodes carry only (id, text, embedding, created_at) - no chunk link.
Propositions attach to entities via ABOUT. So per-chunk coverage is measured through the bridge
Proposition-[:ABOUT]->Entity, Entity.source_chunks. That bridge OVER-attributes (a proposition about a
multi-chunk entity counts for every one of its chunks), so the chunk-bridge gap is a LOWER bound on the
true gap. A sharper direct measure (gold-string present in some proposition text) and a per-entity
measure are reported alongside.

In [1]:
# Imports
import json, datetime, collections
from neo4j import GraphDatabase
import yaml
from rich import print as rprint
NEO4J_URI='bolt://user-konrad.jelen-kgf-neo4j2:7687'  # read-only neo4j2 (NOT .env / live graph)
def norm(x): return ' '.join((x or '').lower().split())

## Gold strings + graph pull

In [2]:
golds=[]
for p in yaml.safe_load(open('../tests/probes/cpap-probe-set.yml')):
    golds+=(p.get('gold_evidence') or [])
golds=list(dict.fromkeys(golds))
rprint(f'{len(golds)} gold evidence strings')

drv=GraphDatabase.driver(NEO4J_URI, auth=('neo4j','kgfoundry'), notifications_min_severity='OFF')
with drv.session() as s:
    chunks=s.run('MATCH (c:Chunk) RETURN c.id AS id, c.text AS text').data()
    # entity -> source_chunks, and whether it has any proposition
    ents=s.run('''MATCH (e:Entity)
                  OPTIONAL MATCH (p:Proposition)-[:ABOUT]->(e)
                  RETURN e.id AS id, e.name AS name, e.source_chunks AS sc,
                         count(p) AS nprop''').data()
    props=[r['text'] for r in s.run('MATCH (p:Proposition) RETURN p.text AS text').data()]
drv.close()
rprint(f'{len(chunks)} chunks, {len(ents)} entities, {len(props)} propositions')

27 gold evidence strings

132 chunks, 2798 entities, 10968 propositions

## Evidence-bearing chunks and their proposition coverage (registered measure)

In [3]:
# chunk -> set of propositions via entity bridge
chunk_has_prop=collections.defaultdict(int)
for e in ents:
    for cid in (e['sc'] or []):
        chunk_has_prop[cid]+=e['nprop']

ev_chunks=[c for c in chunks if any(norm(g) in norm(c['text']) for g in golds)]
ev_ids=[c['id'] for c in ev_chunks]
gap_chunks=[cid for cid in ev_ids if chunk_has_prop.get(cid,0)==0]
gap_frac=len(gap_chunks)/len(ev_ids)
rprint(f'evidence-bearing chunks: {len(ev_ids)} / {len(chunks)}')
rprint(f'[bold]evidence-bearing chunks with ZERO linked propositions (bridge): {len(gap_chunks)} = {gap_frac:.1%}[/bold]')

evidence-bearing chunks: 27 / 132

evidence-bearing chunks with ZERO linked propositions (bridge): 1 = 3.7%

## Direct measure - is each gold string captured in any proposition text

In [4]:
nprop_texts=[norm(t) for t in props]
gold_in_prop={}
for g in golds:
    ng=norm(g)
    gold_in_prop[g]=any(ng in t for t in nprop_texts)
missing_golds=[g for g,v in gold_in_prop.items() if not v]
rprint(f'[bold]gold strings absent from ALL proposition texts: {len(missing_golds)}/{len(golds)} = {len(missing_golds)/len(golds):.1%}[/bold]')
for g in missing_golds: rprint('  MISSING:', repr(g))

gold strings absent from ALL proposition texts: 12/27 = 44.4%

MISSING: '4 to 20 cm H2O'

MISSING: '28 dB(A)'

MISSING: '2,591 m'

MISSING: 'Typical power consumption: 9.0W'

...

MISSING: '290 ml'

MISSING: '238*178*128 mm'

MISSING: '0-60 mins'

MISSING: '26.6 dBA'

MISSING: 'gradually acclimate'

## Per-entity fallback - evidence-bearing entities lacking propositions

In [5]:
# an entity is evidence-bearing if any of its source_chunks contains a gold string
ev_chunk_set=set(ev_ids)
ev_ents=[e for e in ents if set(e['sc'] or []) & ev_chunk_set]
ev_ents_nogold_prop=[e for e in ev_ents if e['nprop']==0]
rprint(f'evidence-bearing entities: {len(ev_ents)}')
rprint(f'evidence-bearing entities with zero propositions: {len(ev_ents_nogold_prop)} = {len(ev_ents_nogold_prop)/max(1,len(ev_ents)):.1%}')
# overall proposition density
nz=sum(1 for e in ents if e['nprop']>0)
rprint(f'entities with >=1 proposition overall: {nz}/{len(ents)} = {nz/len(ents):.1%}')

evidence-bearing entities: 757

evidence-bearing entities with zero propositions: 6 = 0.8%

entities with >=1 proposition overall: 2759/2798 = 98.6%

## Verdict + report

In [6]:
# registered bar keys on the chunk-bridge gap; direct gold-miss reported as the sharper signal
if gap_frac>=0.20: verdict='CONFIRMED'; reason=f'{gap_frac:.1%} of evidence-bearing chunks carry zero propositions (>=20% bar)'
elif gap_frac<0.05: verdict='REFUTED'; reason=f'chunk-bridge coverage saturated ({gap_frac:.1%} gap, <5%)'
else: verdict='INCONCLUSIVE'; reason=f'chunk-bridge gap {gap_frac:.1%} lands in the 5-20% dead zone'
report=dict(hypothesis='R08-H56', scope='deterministic-half-only',
  provenance_note='propositions lack chunk links; chunk gap via ABOUT->entity->source_chunks bridge (over-attributes -> lower bound on gap)',
  n_gold_strings=len(golds), n_chunks=len(chunks), n_evidence_bearing_chunks=len(ev_ids),
  chunk_bridge_zero_prop=len(gap_chunks), chunk_bridge_gap_frac=gap_frac, gap_chunk_ids=gap_chunks,
  gold_strings_absent_from_all_props=len(missing_golds), gold_miss_frac=len(missing_golds)/len(golds), missing_golds=missing_golds,
  evidence_bearing_entities=len(ev_ents), ev_entities_zero_prop=len(ev_ents_nogold_prop),
  entities_with_prop_overall=nz, entity_prop_coverage=nz/len(ents),
  bar='>=20% chunk gaps confirms; <5% refutes', verdict=verdict, reason=reason)
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
path=f'../reports/proposition-coverage-h56-{stamp}.json'
json.dump(report,open(path,'w'),indent=2)
rprint(f'[bold green]{verdict}[/bold green] - {reason}')
rprint('wrote',path)

REFUTED - chunk-bridge coverage saturated (3.7% gap, <5%)

wrote ../reports/proposition-coverage-h56-20260707-092753.json